In [1]:
import numpy as np
from numpy.linalg import inv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2
sns.set()

from sklearn.datasets import make_spd_matrix
from sklearn.covariance import graphical_lasso, GraphicalLasso, GraphicalLassoCV
from tqdm.notebook import tqdm

from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import scipy.cluster.hierarchy as sch

%matplotlib inline

# fix random seed
np.random.seed(40)

In [2]:
def is_pos_def(A):
    if is_symmetric(A):
        try:
            np.linalg.cholesky(A)
            return True
        except np.linalg.LinAlgError:
            return False
    else:
        return False

def is_symmetric(a, tol=1e-8):
    return np.all(np.abs(a-a.T) < tol)

In [3]:
def gradient_descent_step(coeffs_zero, H_s, C, M, alpha):
    coeffs_new = np.zeros(coeffs_zero.shape)
    psi_hat = (coeffs_zero.reshape(-1, 1, 1)*H_s).sum(0)
    diff = 0
    for h in range(M):
        grad_log_l_h = np.trace(inv(psi_hat).dot(H_s[h])) - np.trace(H_s[h].dot(C))
        diff += np.abs(alpha*grad_log_l_h)
        coeffs_new[h] = coeffs_zero[h] + alpha*grad_log_l_h
        
    return coeffs_new, diff

def gradient_descent(coeffs_zero, H_s, C, M, alpha=0.01, iters=15):
    coeffs_imo = coeffs_zero
    for i in range(iters):
        print(coeffs_imo)
        coeffs_imo, diff = gradient_descent_step(coeffs_imo, H_s, C, M, alpha=alpha)
        #print(diff)
        if diff < 1e-5:
            break
    #print(coeffs_imo)
    
    return coeffs_imo

def gradient_descent_step_single(coeffs_zero, H_s, C, M, alpha, to_change=None):
    coeffs_new = np.zeros(coeffs_zero.shape)
    psi_hat = (coeffs_zero.reshape(-1, 1, 1)*H_s).sum(0)
    diff = 0
    for h in range(M):
        if to_change == None or to_change == h:
            grad_log_l_h = np.trace(inv(psi_hat).dot(H_s[h])) - np.trace(H_s[h].dot(C))
            diff += np.abs(alpha*grad_log_l_h)
            coeffs_new[h] = coeffs_zero[h] + alpha*grad_log_l_h
        else:
            coeffs_new[h] = coeffs_zero[h]
        
    return coeffs_new, diff

def gradient_descent_single(coeffs_zero, H_s, C, M, alpha=0.01, iters=15, to_change=None):
    coeffs_imo = coeffs_zero
    for i in range(iters):
        #print(coeffs_imo)
        coeffs_imo, diff = gradient_descent_step_single(coeffs_imo, H_s, C, M, alpha=alpha, to_change=to_change)
        #print(diff)
        if diff < 1e-5:
            break
    #print(coeffs_imo)
    
    return coeffs_imo

def calc_matrices_precision(psi_hat, H_s, C, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim,))
    inv_psi_hat = inv(psi_hat) # calculate psi_hat inverse
    for g in range(dim):
        for h in range(dim):
            mult = ((inv_psi_hat.dot(H_s[g])).dot(inv_psi_hat)).dot(H_s[h])
            lhs = np.trace(mult)
            A[g, h] = lhs
        rhs = np.trace(inv_psi_hat.dot(H_s[g])) - np.trace(C.dot(H_s[g]))
        B[g] = rhs
        
    return A, B

def iterative_soln_precision(coeffs_zero, H_s, C, dim, iters=5):
    s_imo = coeffs_zero
    for it in range(iters):
        psi_hat = (s_imo.reshape(-1, 1, 1)*H_s).sum(0) # calculate psi_hat
        A, B = calc_matrices_precision(psi_hat, H_s, C, dim)
        t_i = inv(A).dot(B)
        s_i = s_imo + t_i
        print(s_i)
        s_imo = s_i
        
    return s_imo

def iterative_soln_precision_single(coeffs_zero, H_s, C, dim, modify_index=0, iters=5):
    s_imo = coeffs_zero
    for it in range(iters):
        psi_hat = (s_imo.reshape(-1, 1, 1)*H_s).sum(0) # calculate psi_hat
        A, B = calc_matrices_precision(psi_hat, H_s, C, dim)
        t_i = inv(A).dot(B)
        s_i = s_imo.copy()
        s_i[modify_index] = s_imo[modify_index] + t_i[modify_index]
        #print(s_i)
        s_imo = s_i
        
    return s_imo

def unbiased_init_precision(C, H_s, N, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim, ))
    for g in range(dim):
        for h in range(dim):
            A[g, h] = np.trace(H_s[g].dot(H_s[h]))
        B[g] = np.trace(inv(C).dot(H_s[g]))
    return inv(A).dot(B)

def calc_likelihood_precision(coeffs, H_s, C, N, P):
    precision_hat = (coeffs.reshape(-1, 1, 1)*H_s).sum(0)
    log_l = -P*np.log(2*np.pi) + np.log(np.linalg.det(precision_hat)) - np.trace(np.dot(precision_hat, C))
    
    return log_l*(N/2)

def generate_matrices(prec_coeffs=None, M=2, dim=4):
    H_s = []
    if not prec_coeffs:
        prec_coeffs = np.random.rand(M)
    precision = np.zeros((dim, dim))
    H_s_stacked = np.zeros((dim**2, M))
    for i in range(M):
        mat = make_spd_matrix(dim)
        H_s.append(mat)
        precision += prec_coeffs[i]*mat
        H_s_stacked[:, i] = mat.flatten()
    H_s = np.array(H_s)
    
    # ensure basis matrices are linearly independent
    assert np.linalg.matrix_rank(H_s_stacked) == M, "Not Linearly Independent basis matrices"
    
    # ensure it's actually symmetric
    # minimal modification on scale of 1e-15
    precision_corrected = (precision + precision.T)/2
    
    return H_s, precision_corrected, prec_coeffs

def collect_precision_matrix(H_s, prec_coeffs):
    precision = (prec_coeffs.reshape(-1, 1, 1)*H_s).sum(0)
    
    return precision

def sim_data(covar, dim, N=1000):
    assert is_symmetric(covar), is_pos_def(covar)
    data_sim = np.random.multivariate_normal(np.zeros(dim), covar, N).T
    data_sim = data_sim - data_sim.mean()
    C = np.cov(data_sim)
    
    return data_sim, C

def likelihood_ratio_test(likelihood_null, likelihood_alternative, dof):
    delta_d = -2*(likelihood_null-likelihood_alternative)
    
    return delta_d, chi2.pdf(delta_d, dof)

def cluster_precision(precision):
    dist_mat = np.abs(precision).max() - np.abs(precision)
    np.fill_diagonal(dist_mat, 0)
    assert is_symmetric(dist_mat), is_pos_def(dist_mat)
    
    #pairwise_distances = sch.distance.pdist(precision_one)
    pairwise_distances = squareform(dist_mat)
    Z = linkage(pairwise_distances,
                method='average'  # dissimilarity metric: max distance across all pairs of 
                                   # records between two clusters
        )

    # calculate full dendrogram and visualize it
    plt.figure(figsize=(10, 5))
    dendrogram(
                Z,
                orientation='right',
                labels=np.arange(dim),
                distance_sort='descending',
                show_leaf_counts=False
              )
    plt.show()
    
    return Z

In [4]:
M = 2
dim = 4
N = 500

H_s, precision_one, prec_coeffs_one = generate_matrices(M=M, dim=dim)
data_one, C_one = sim_data(covar=inv(precision_one), dim=dim, N=N)

In [5]:
s_zero_one = unbiased_init_precision(C_one, H_s, N=N, dim=M)
coeffs_hat_one = iterative_soln_precision(s_zero_one, H_s, C_one, dim=M, iters=5)
coeffs_hat_one, prec_coeffs_one

[0.41444528 0.05295547]
[0.41584342 0.05282875]
[0.41584771 0.05282836]
[0.41584771 0.05282836]
[0.41584771 0.05282836]


(array([0.41584771, 0.05282836]), array([0.40768703, 0.05536604]))

In [6]:
coeffs_hat_one_gd = gradient_descent_single(s_zero_one, H_s, C_one, M=M, alpha=0.01, iters=500, to_change=0)
s_zero_one, coeffs_hat_one_gd, prec_coeffs_one

(array([0.44119729, 0.05059454]),
 array([0.41851253, 0.05059454]),
 array([0.40768703, 0.05536604]))

In [7]:
prec_coeffs_two = prec_coeffs_one.copy()
prec_coeffs_two[0] += 0.1
precision_two = collect_precision_matrix(H_s, prec_coeffs_two)
data_two, C_two = sim_data(covar=inv(precision_two), dim=dim, N=N)

In [8]:
s_zero_two = unbiased_init_precision(C_two, H_s, N=N, dim=M)
coeffs_hat_two = iterative_soln_precision(s_zero_two, H_s, C_two, dim=M, iters=10)
coeffs_hat_two, prec_coeffs_two

[0.49226169 0.0663845 ]
[0.49908677 0.06576779]
[0.49917341 0.06575987]
[0.49917342 0.06575986]
[0.49917342 0.06575986]
[0.49917342 0.06575986]
[0.49917342 0.06575986]
[0.49917342 0.06575986]
[0.49917342 0.06575986]
[0.49917342 0.06575986]


(array([0.49917342, 0.06575986]), array([0.50768703, 0.05536604]))

In [9]:
data_total = np.concatenate((data_one, data_two), axis=1)
C_total = np.cov(data_total)
s_zero_total = unbiased_init_precision(C_total, H_s, N=N, dim=M)
coeffs_hat_total = iterative_soln_precision(s_zero_total, H_s, C_total, dim=M, iters=10)
coeffs_hat_total

[0.45047523 0.05902766]
[0.45324663 0.05877775]
[0.45326215 0.05877633]
[0.45326215 0.05877633]
[0.45326215 0.05877633]
[0.45326215 0.05877633]
[0.45326215 0.05877633]
[0.45326215 0.05877633]
[0.45326215 0.05877633]
[0.45326215 0.05877633]


array([0.45326215, 0.05877633])

In [10]:
"""
LRT Using total data to fit null model
"""

# covariance from total data/model - sample size 2*N
null_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_total, H_s=H_s, C=C_total, 
                                            N=2*N, P=dim)
# covariance from first data, total model
alt_likelihood_one = calc_likelihood_precision(coeffs=coeffs_hat_total, H_s=H_s, C=C_one,
                                               N=N, P=dim)
# covariance from second model, second data
alt_likelihood_two = calc_likelihood_precision(coeffs=coeffs_hat_two, H_s=H_s, C=C_two, 
                                              N=N, P=dim)

likelihood_ratio_test(null_likelihood, alt_likelihood_one+alt_likelihood_two, M)

(10.445141240878002, 0.0026967233932131303)

In [11]:
"""
LRT Using total data to fit null model

Fit two models in denominator
"""

# covariance from total data/model - sample size 2*N
null_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_total, H_s=H_s, C=C_total, 
                                            N=2*N, P=dim)
# covariance from first data, total model
alt_likelihood_one = calc_likelihood_precision(coeffs=coeffs_hat_one, H_s=H_s, C=C_one,
                                               N=N, P=dim)
# covariance from second model, second data
alt_likelihood_two = calc_likelihood_precision(coeffs=coeffs_hat_two, H_s=H_s, C=C_two, 
                                              N=N, P=dim)

likelihood_ratio_test(null_likelihood, alt_likelihood_one+alt_likelihood_two, M)

(18.854430005070753, 4.025154446255738e-05)

In [12]:
"""
Refit model on second data using FIRST model's init

Run two LRT - one for each coefficient change
"""
# covariance from total data/model - sample size 2*N
null_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_total, H_s=H_s, C=C_total, 
                                            N=2*N, P=dim)
# modify the first index of coeffs, use second data
mod_first_coeffs = iterative_soln_precision_single(coeffs_hat_one, H_s, 
                                                   C=C_two, dim=M, modify_index=0, iters=15)
# modify the second index of coeffs, use second data
mod_second_coeffs = iterative_soln_precision_single(coeffs_hat_one, H_s, 
                                                    C=C_two, dim=M, modify_index=1, iters=15)
# covariance from first model, first data
first_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_one, H_s=H_s, C=C_one, N=N, P=dim)
# covariance from second model, second data, modified first coefficient
alt_likelihood_first = calc_likelihood_precision(coeffs=mod_first_coeffs, H_s=H_s, 
                                                     C=C_two, N=N, P=dim)
# covariance from second model, second data, modified second coefficient
alt_likelihood_second = calc_likelihood_precision(coeffs=mod_second_coeffs, H_s=H_s, 
                                                      C=C_two, N=N, P=dim)
first_test = likelihood_ratio_test(null_likelihood, 
                                   first_likelihood+alt_likelihood_first, M)
second_test = likelihood_ratio_test(null_likelihood,
                                   first_likelihood+alt_likelihood_second, M)
first_test, second_test

((14.910989769374282, 0.0002891277097400726), (-2.3379294309706893, 0.0))

In [13]:
alt_likelihood_first, alt_likelihood_second, null_likelihood/2

(-3655.6169558180877, -3664.24141541826, -3753.260020152333)

In [14]:
coeffs_hat_one, mod_first_coeffs, mod_second_coeffs, prec_coeffs_one, prec_coeffs_two

(array([0.41584771, 0.05282836]),
 array([0.49983823, 0.05282836]),
 array([0.41584771, 0.06657242]),
 array([0.40768703, 0.05536604]),
 array([0.50768703, 0.05536604]))

In [15]:
"""
Refit model on second data using FIRST model's init, using first order iterative method

Run two LRT - one for each coefficient change
"""
# covariance from total data/model - sample size 2*N
null_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_total, H_s=H_s, C=C_total, 
                                            N=2*N, P=dim)
# modify the first index of coeffs, use second data
mod_first_coeffs = gradient_descent_single(coeffs_hat_one, H_s, C_two, M=M, alpha=0.01, iters=500, to_change=0)
# modify the second index of coeffs, use second data
mod_second_coeffs = gradient_descent_single(coeffs_hat_one, H_s, C_two, M=M, alpha=0.01, iters=500, to_change=1)
# covariance from first model, first data
first_likelihood = calc_likelihood_precision(coeffs=coeffs_hat_one, H_s=H_s, C=C_one, N=N, P=dim)
# covariance from second model, second data, modified first coefficient
alt_likelihood_first = calc_likelihood_precision(coeffs=mod_first_coeffs, H_s=H_s, 
                                                     C=C_two, N=N, P=dim)
# covariance from second model, second data, modified second coefficient
alt_likelihood_second = calc_likelihood_precision(coeffs=mod_second_coeffs, H_s=H_s, 
                                                      C=C_two, N=N, P=dim)
first_test = likelihood_ratio_test(null_likelihood, 
                                   first_likelihood+alt_likelihood_first, M)
second_test = likelihood_ratio_test(null_likelihood,
                                   first_likelihood+alt_likelihood_second, M)
first_test, second_test

((15.54257847894769, 0.00021083464176841038),
 (0.830513594988588, 0.33008536437335356))

In [16]:
alt_likelihood_first, alt_likelihood_second, null_likelihood/2

(-3655.301161463301, -3662.6571939052806, -3753.260020152333)

In [17]:
coeffs_hat_one, mod_first_coeffs, mod_second_coeffs, prec_coeffs_one, prec_coeffs_two

(array([0.41584771, 0.05282836]),
 array([0.51478951, 0.05282836]),
 array([0.41584771, 0.07859972]),
 array([0.40768703, 0.05536604]),
 array([0.50768703, 0.05536604]))

In [18]:
# mle(total data) -> mle(first) + mle(second) for single coeff

In [19]:
data_one.mean(), data_two.mean(), data_total.mean()

(-5.329070518200751e-17, 0.0, -2.6645352591003756e-17)